# PSSN Week 2 — Semantic-state quality validation

Run in Colab after notebook 06 has created `semantic_states.jsonl`. The validator checks each trace's states without rewriting them. Short (<3-token) or structurally malformed sequences are excluded from the Week-3-ready file; long (>512-token) states are retained but flagged for manual review.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")
print("Repository:", Path.cwd())

In [ ]:
try:
    from src.semantic_states.validation import validate_semantic_state_records
    print("Semantic-state validator is available in the checkout.")
except ImportError:
    from google.colab import files
    import re
    uploaded = files.upload()
    required = {"__init__.py", "schema.py", "segmenter.py", "tokenizer_utils.py", "validation.py"}
    normalized_uploads = {}
    for uploaded_name, content in uploaded.items():
        canonical_name = re.sub(r" \(\d+\)(?=\.py$)", "", Path(uploaded_name).name)
        if canonical_name in required:
            normalized_uploads[canonical_name] = content
    missing = required - set(normalized_uploads)
    if missing:
        raise ValueError(f"Upload all five semantic_states files; missing: {sorted(missing)}")
    package_dir = Path("src/semantic_states")
    package_dir.mkdir(parents=True, exist_ok=True)
    for filename in required:
        (package_dir / filename).write_bytes(normalized_uploads[filename])
    from src.semantic_states.validation import validate_semantic_state_records
    print("Uploaded the current semantic-state validator and dependencies.")

In [ ]:
from datetime import datetime, timezone
import json

ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts/expanded_baseline")
MIN_STATE_TOKENS = 3
MIN_USABLE_STATES = 2
MAX_STATE_TOKENS_WARNING = 512

candidate_files = sorted(ARTIFACT_ROOT.glob("*/semantic_state_segmentation/*/semantic_states.jsonl"))
if not candidate_files:
    raise FileNotFoundError(f"No segmented semantic_states.jsonl found under {ARTIFACT_ROOT}. Run notebook 06 first.")
segmented_path = candidate_files[-1]
run_dir = segmented_path.parents[2]
source_path = run_dir / "segmentation_ready_traces.jsonl"
if not source_path.is_file():
    raise FileNotFoundError(f"Original trace file is missing: {source_path}")
state_records = [json.loads(line) for line in segmented_path.read_text(encoding="utf-8").splitlines() if line.strip()]
source_records = [json.loads(line) for line in source_path.read_text(encoding="utf-8").splitlines() if line.strip()]
source_traces = {}
for record in source_records:
    source_traces[f"{record.get('run_id')}:{record.get('problem_id')}"] = record.get("reasoning_trace", "")
    if record.get("problem_id"):
        source_traces[record["problem_id"]] = record.get("reasoning_trace", "")
print(f"Input: {segmented_path} ({len(state_records)} states)")

result = validate_semantic_state_records(
    state_records,
    source_traces=source_traces,
    min_state_tokens=MIN_STATE_TOKENS,
    min_usable_states=MIN_USABLE_STATES,
    max_state_tokens=MAX_STATE_TOKENS_WARNING,
)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = segmented_path.parent / "quality_validation" / stamp
output_dir.mkdir(parents=True, exist_ok=False)
reports_path = output_dir / "trace_validation_reports.jsonl"
issues_path = output_dir / "validation_issues.jsonl"
week3_path = output_dir / "week3_ready_states.jsonl"
summary_path = output_dir / "validation_summary.json"
reports_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in result["trace_reports"]), encoding="utf-8")
issues = [
    {"trace_id": report["trace_id"], "problem_id": report["problem_id"], **issue}
    for report in result["trace_reports"] for issue in report["issues"]
]
issues_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in issues), encoding="utf-8")
week3_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in result["week3_ready_states"]), encoding="utf-8")
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_segmented_file": str(segmented_path),
    "source_trace_file": str(source_path),
    **{key: result[key] for key in ("trace_count", "valid_trace_count", "week3_ready_trace_count", "invalid_trace_count", "state_count", "week3_ready_state_count", "warning_count", "thresholds")},
    "output_files": {"reports": str(reports_path), "issues": str(issues_path), "week3_ready_states": str(week3_path)},
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
print("Invalid trace examples:")
for report in result["trace_reports"]:
    if not report["is_valid"]:
        print(report["problem_id"], report["issues"][:4])
print("\nManual-review warnings:")
for report in result["trace_reports"]:
    warnings = [issue for issue in report["issues"] if issue["severity"] == "warning"]
    if warnings:
        print(report["problem_id"], warnings)
print(f"\nWeek-3-ready: {result['week3_ready_trace_count']} traces / {result['week3_ready_state_count']} states")